# Training Curves Comparison

This notebook compares training curves across different model architectures:
- **Structure-aware GIN**: Uses secondary structure features
- **Sequence-based GNN baselines**: GAT, GCN, GIN (sequence only)
- **Sequence-based Transformer baseline**

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from scipy.ndimage import uniform_filter1d

# Set publication-quality style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("paper", font_scale=1.3)
sns.set_palette("husl")

# Enhanced font settings for electronic publication (matching scaling law plot)
plt.rcParams['figure.dpi'] = 150  # Higher DPI for screen display
plt.rcParams['savefig.dpi'] = 600  # Publication quality (600 DPI for electronic)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.titlesize'] = 15
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['savefig.format'] = 'png'
plt.rcParams['savefig.bbox'] = 'tight'
plt.rcParams['savefig.pad_inches'] = 0.1

In [ ]:
# Define paths to experiment results
experiments = {
    'Structure-aware GIN': '/home/jeff/ribozyme/results/runs/gin_True_True_True_True_20251210_184823/history.json',
    'Seq-based GAT': '/home/jeff/ribozyme/results/gnn_seq_baseline/gat_seq_only_20251211_132438/gat_True_False_False_False_20251211_132438/history.json',
    'Seq-based GCN': '/home/jeff/ribozyme/results/gnn_seq_baseline/gcn_seq_only_20251211_130853/gcn_True_False_False_False_20251211_130853/history.json',
    'Seq-based GIN': '/home/jeff/ribozyme/results/gnn_seq_baseline/gin_seq_only_20251211_134349/gin_True_False_False_False_20251211_134349/history.json',
    'Seq-based Transformer': '/home/jeff/ribozyme/results/seq_transformer_baseline/seq_transformer_fast_20251211_121145/history.json'
}

In [ ]:
# Load training histories
histories = {}
display_names = {
    'Structure-aware GIN': 'Structure-aware GIN',
    'Seq-based GAT': 'Seq-based GAT Baseline',
    'Seq-based GCN': 'Seq-based GCN Baseline',
    'Seq-based GIN': 'Seq-based GIN Baseline',
    'Seq-based Transformer': 'Seq-based Transformer Baseline'
}

for name, path in experiments.items():
    try:
        with open(path, 'r') as f:
            data = json.load(f)
            # Store with display name
            display_name = display_names[name]
            histories[display_name] = data
        print(f"✓ Loaded {display_name}: {len(data['val_acc'])} epochs")
    except FileNotFoundError:
        print(f"✗ File not found: {name} at {path}")
    except Exception as e:
        print(f"✗ Error loading {name}: {e}")

In [ ]:
# Define colors and styles for each model
colors = {
    'Structure-aware GIN': '#E74C3C',  # Red - highlight the main model
    'Seq-based GAT Baseline': '#3498DB',        # Blue
    'Seq-based GCN Baseline': '#9B59B6',        # Purple
    'Seq-based GIN Baseline': '#F39C12',        # Orange
    'Seq-based Transformer Baseline': '#2ECC71' # Green
}

linestyles = {
    'Structure-aware GIN': '-',        # Solid for main model
    'Seq-based GAT Baseline': '--',             # Dashed for baselines
    'Seq-based GCN Baseline': '--',
    'Seq-based GIN Baseline': '--',
    'Seq-based Transformer Baseline': '-.'
}

linewidths = {
    'Structure-aware GIN': 3,          # Thicker for main model
    'Seq-based GAT Baseline': 2,
    'Seq-based GCN Baseline': 2,
    'Seq-based GIN Baseline': 2,
    'Seq-based Transformer Baseline': 2.5
}

In [ ]:
# Smoothing function for loss curves
def smooth_curve(values, window_size=5):
    """Apply moving average smoothing to remove spikes"""
    return uniform_filter1d(values, size=window_size, mode='nearest')

# Create 2x2 grid of plots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# ==================== TOP ROW: TRAINING METRICS ====================

# Plot 1 (Top Left): Training Accuracy
for name, history in histories.items():
    epochs = range(1, len(history['train_acc']) + 1)
    ax1.plot(epochs, history['train_acc'],
             label=name,
             color=colors[name],
             linestyle=linestyles[name],
             linewidth=linewidths[name],
             alpha=0.9)

ax1.set_xlabel('Epoch', fontsize=14, fontweight='bold')
ax1.set_ylabel('Training Accuracy', fontsize=14, fontweight='bold')
ax1.set_title('Training Accuracy vs Epoch', fontsize=16, fontweight='bold', pad=20)
ax1.grid(True, alpha=0.3, linestyle='--', linewidth=0.8, color='gray')
ax1.set_axisbelow(True)

# Format y-axis as percentage
def percent_formatter(y, pos):
    return f'{y:.1%}'
ax1.yaxis.set_major_formatter(FuncFormatter(percent_formatter))

ax1.set_facecolor('#F0F8FF')  # Light blue background for training
ax1.minorticks_on()
ax1.grid(which='minor', alpha=0.15, linestyle=':', linewidth=0.5)

# Plot 2 (Top Right): Training Loss (smoothed)
for name, history in histories.items():
    epochs = range(1, len(history['train_loss']) + 1)
    train_loss = np.array(history['train_loss'])
    
    # Apply smoothing to loss curves to remove spikes
    smoothed_loss = smooth_curve(train_loss, window_size=5)
    
    ax2.plot(epochs, smoothed_loss,
             color=colors[name],
             linestyle=linestyles[name],
             linewidth=linewidths[name],
             alpha=0.9)  # Same alpha as accuracy plots

ax2.set_xlabel('Epoch', fontsize=14, fontweight='bold')
ax2.set_ylabel('Training Loss (Smoothed)', fontsize=14, fontweight='bold')
ax2.set_title('Training Loss vs Epoch', fontsize=16, fontweight='bold', pad=20)
ax2.set_ylim(0, 3)
ax2.grid(True, alpha=0.3, linestyle='--', linewidth=0.8, color='gray')
ax2.set_axisbelow(True)

ax2.set_facecolor('#F0F8FF')  # Light blue background for training
ax2.minorticks_on()
ax2.grid(which='minor', alpha=0.15, linestyle=':', linewidth=0.5)

# ==================== BOTTOM ROW: VALIDATION METRICS ====================

# Plot 3 (Bottom Left): Validation Accuracy
for name, history in histories.items():
    epochs = range(1, len(history['val_acc']) + 1)
    ax3.plot(epochs, history['val_acc'],
             color=colors[name],
             linestyle=linestyles[name],
             linewidth=linewidths[name],
             alpha=0.9)

ax3.set_xlabel('Epoch', fontsize=14, fontweight='bold')
ax3.set_ylabel('Validation Accuracy', fontsize=14, fontweight='bold')
ax3.set_title('Validation Accuracy vs Epoch', fontsize=16, fontweight='bold', pad=20)
ax3.grid(True, alpha=0.3, linestyle='--', linewidth=0.8, color='gray')
ax3.set_axisbelow(True)
ax3.yaxis.set_major_formatter(FuncFormatter(percent_formatter))

ax3.set_facecolor('#FFF5F0')  # Light orange/peach background for validation
ax3.minorticks_on()
ax3.grid(which='minor', alpha=0.15, linestyle=':', linewidth=0.5)

# Plot 4 (Bottom Right): Validation Loss (smoothed)
for name, history in histories.items():
    epochs = range(1, len(history['val_loss']) + 1)
    val_loss = np.array(history['val_loss'])
    
    smoothed_loss = smooth_curve(val_loss, window_size=5)
    
    ax4.plot(epochs, smoothed_loss,
             color=colors[name],
             linestyle=linestyles[name],
             linewidth=linewidths[name],
             alpha=0.9)  # Same alpha as accuracy plots

ax4.set_xlabel('Epoch', fontsize=14, fontweight='bold')
ax4.set_ylabel('Validation Loss (Smoothed)', fontsize=14, fontweight='bold')
ax4.set_title('Validation Loss vs Epoch', fontsize=16, fontweight='bold', pad=20)
ax4.set_ylim(0, 3)
ax4.grid(True, alpha=0.3, linestyle='--', linewidth=0.8, color='gray')
ax4.set_axisbelow(True)

ax4.set_facecolor('#FFF5F0')  # Light orange/peach background for validation
ax4.minorticks_on()
ax4.grid(which='minor', alpha=0.15, linestyle=':', linewidth=0.5)

# Add a single legend for the entire figure (no title, more space from plots)
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, 
           loc='upper center', 
           bbox_to_anchor=(0.5, 0.995),  # Even further from plots (was 0.985)
           ncol=5,
           fontsize=14,
           frameon=True,
           fancybox=True,
           shadow=True,
           markerscale=1.5)

fig.patch.set_facecolor('white')
plt.tight_layout(rect=[0, 0, 1, 0.93])  # More space for legend (was 0.94)
plt.show()

# Save figure
fig.savefig('/home/jeff/ribozyme/notebooks/training_curves_comparison.png',
            dpi=600,
            bbox_inches='tight',
            facecolor='white',
            edgecolor='none')
print("Figure saved as 'training_curves_comparison.png' at 600 DPI")

# Print peak information for loss curves
print("\n" + "="*80)
print("LOSS CURVE PEAKS")
print("="*80)
for name, history in histories.items():
    val_loss = np.array(history['val_loss'])
    peak_idx = np.argmax(val_loss)
    peak_value = val_loss[peak_idx]
    print(f"{name}: Peak at epoch {peak_idx+1}, value = {peak_value:.4f}")

In [ ]:
# Print summary statistics
print("\n" + "="*80)
print("FINAL PERFORMANCE SUMMARY")
print("="*80)

results = []
for name, history in histories.items():
    final_val_acc = history['val_acc'][-1]
    final_val_loss = history['val_loss'][-1]
    best_val_acc = max(history['val_acc'])
    best_epoch = history['val_acc'].index(best_val_acc) + 1
    results.append((name, final_val_acc, final_val_loss, best_val_acc, best_epoch))

# Sort by best validation accuracy
results.sort(key=lambda x: x[3], reverse=True)

print(f"\n{'Model':<30} {'Final Val Acc':<15} {'Final Val Loss':<15} {'Best Val Acc':<15} {'Best Epoch'}")
print("-" * 100)
for name, final_acc, final_loss, best_acc, best_epoch in results:
    print(f"{name:<30} {final_acc:>14.4f} {final_loss:>14.4f} {best_acc:>14.4f} {best_epoch:>11}")

print("\n" + "="*80)

In [ ]:
# Calculate improvement of structure-aware model over baselines
if 'Structure-aware GIN' in histories:
    structure_acc = max(histories['Structure-aware GIN']['val_acc'])
    
    print("\n" + "="*80)
    print("STRUCTURE-AWARE GIN vs BASELINES")
    print("="*80)
    
    for name, history in histories.items():
        if name != 'Structure-aware GIN':
            baseline_acc = max(history['val_acc'])
            improvement = structure_acc - baseline_acc
            pct_improvement = (improvement / baseline_acc) * 100
            print(f"\nvs {name}:")
            print(f"  Absolute improvement: {improvement:+.4f}")
            print(f"  Relative improvement: {pct_improvement:+.2f}%")